# Diwali Sales Analysis
### Exploratory data analysis with Python

## Project objective
Analyze Diwali sales records to understand customer purchasing patterns and identify high-performing customer segments, states, occupations, and product categories. The analysis is descriptive: it highlights patterns in this dataset and does not establish causation.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Jupyter Notebook.

## 1. Import libraries

The notebook uses a project-relative data path so it can run on another computer after the repository is cloned.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", None)

# Works when Jupyter is launched from either the repository root or notebooks/.
possible_paths = [
    Path("data/diwali_sales_data.csv"),
    Path("../data/diwali_sales_data.csv"),
]
data_path = next((path for path in possible_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Dataset not found. Place diwali_sales_data.csv in the repository's data/ folder."
    )

df = pd.read_csv(data_path)
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns.")

## 2. Understand the data

Review the sample, data types, summary statistics, and missing values before analysis.

In [ ]:
display(df.head())
df.info()
display(df.describe(include="all").T)

In [ ]:
missing_summary = (
    df.isna().sum()
      .rename("missing_values")
      .to_frame()
      .query("missing_values > 0")
      .sort_values("missing_values", ascending=False)
)
display(missing_summary if not missing_summary.empty else "No missing values found.")

## 3. Data cleaning

The source file contains 12 missing values in `Amount`. Those rows remain available for analyses that do not require a sales amount. For sales-value analysis, use a separate DataFrame that excludes rows where `Amount` is missing. This avoids inventing sales values or dropping records unnecessarily.

In [ ]:
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
df["Orders"] = pd.to_numeric(df["Orders"], errors="coerce")

df_sales = df.dropna(subset=["Amount"]).copy()

print(f"All records: {len(df):,}")
print(f"Records with a valid Amount: {len(df_sales):,}")
print(f"Records excluded from sales-value analysis due to missing Amount: {df['Amount'].isna().sum():,}")

## 4. Customer demographics

### Gender
Compare customer-record counts and total recorded sales amount by gender.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

gender_counts = df["Gender"].value_counts().reindex(["F", "M"])
axes[0].bar(gender_counts.index.astype(str), gender_counts.values)
axes[0].set(title="Customer records by gender", xlabel="Gender", ylabel="Records")

gender_sales = df_sales.groupby("Gender", as_index=False)["Amount"].sum().sort_values("Amount", ascending=False)
axes[1].bar(gender_sales["Gender"].astype(str), gender_sales["Amount"])
axes[1].set(title="Recorded sales amount by gender", xlabel="Gender", ylabel="Amount")

plt.tight_layout()
plt.show()
display(gender_sales)

### Age group

In [ ]:
age_sales = (
    df_sales.groupby("Age Group", as_index=False)["Amount"].sum()
    .sort_values("Amount", ascending=False)
)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(age_sales["Age Group"].astype(str), age_sales["Amount"])
ax.set(title="Recorded sales amount by age group", xlabel="Age group", ylabel="Amount")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
display(age_sales)

## 5. Geographic analysis

Compare states by number of orders and total recorded sales amount. `Orders` is summed as supplied in the dataset; it should not automatically be interpreted as unique transactions without a data dictionary.

In [ ]:
state_orders = (
    df.groupby("State", as_index=False)["Orders"].sum()
    .sort_values("Orders", ascending=False).head(10)
)
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(state_orders["State"].astype(str), state_orders["Orders"])
ax.set(title="Top 10 states by summed orders", xlabel="State", ylabel="Summed orders")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
display(state_orders)

In [ ]:
state_sales = (
    df_sales.groupby("State", as_index=False)["Amount"].sum()
    .sort_values("Amount", ascending=False).head(10)
)
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(state_sales["State"].astype(str), state_sales["Amount"])
ax.set(title="Top 10 states by recorded sales amount", xlabel="State", ylabel="Amount")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
display(state_sales)

## 6. Marital status and occupation

`Marital_Status` is encoded numerically in the source data. The notebook preserves the source values rather than assuming a label mapping without a data dictionary.

In [ ]:
marital_sales = (
    df_sales.groupby(["Marital_Status", "Gender"], as_index=False)["Amount"].sum()
    .sort_values("Amount", ascending=False)
)
fig, ax = plt.subplots(figsize=(8, 4))

for i, gender in enumerate(sorted(marital_sales["Gender"].dropna().unique())):
    subset = marital_sales[marital_sales["Gender"] == gender].set_index("Marital_Status").reindex(sorted(marital_sales["Marital_Status"].unique()))
    positions = np.arange(len(subset)) + (i - 0.5) * 0.35
    ax.bar(positions, subset["Amount"], width=0.35, label=str(gender))
ax.set_xticks(np.arange(len(sorted(marital_sales["Marital_Status"].unique()))))
ax.set_xticklabels(sorted(marital_sales["Marital_Status"].unique()))
ax.legend(title="Gender")
ax.set(title="Recorded sales by marital-status code and gender", xlabel="Marital-status code", ylabel="Amount")
plt.tight_layout()
plt.show()
display(marital_sales)

In [ ]:
occupation_sales = (
    df_sales.groupby("Occupation", as_index=False)["Amount"].sum()
    .sort_values("Amount", ascending=False)
)
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(occupation_sales["Occupation"].astype(str), occupation_sales["Amount"])
ax.set(title="Recorded sales amount by occupation", xlabel="Occupation", ylabel="Amount")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()
display(occupation_sales)

## 7. Product-category analysis

In [ ]:
category_sales = (
    df_sales.groupby("Product_Category", as_index=False)["Amount"].sum()
    .sort_values("Amount", ascending=False)
)
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(category_sales.head(10)["Product_Category"].astype(str), category_sales.head(10)["Amount"])
ax.set(title="Top 10 product categories by recorded sales amount", xlabel="Product category", ylabel="Amount")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()
display(category_sales.head(10))

In [ ]:
top_products = (
    df.groupby("Product_ID", as_index=False)["Orders"].sum()
    .sort_values("Orders", ascending=False).head(10)
)
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(top_products["Product_ID"].astype(str), top_products["Orders"])
ax.set(title="Top 10 product IDs by summed orders", xlabel="Product ID", ylabel="Summed orders")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
display(top_products)

## 8. Key findings

The cells below calculate the rankings directly from the data rather than hard-coding conclusions.

In [ ]:
total_sales = df_sales["Amount"].sum()
gender_sales_rank = df_sales.groupby("Gender")["Amount"].sum().sort_values(ascending=False)
age_sales_rank = df_sales.groupby("Age Group")["Amount"].sum().sort_values(ascending=False)
state_sales_rank = df_sales.groupby("State")["Amount"].sum().sort_values(ascending=False)
category_sales_rank = df_sales.groupby("Product_Category")["Amount"].sum().sort_values(ascending=False)
occupation_sales_rank = df_sales.groupby("Occupation")["Amount"].sum().sort_values(ascending=False)

print(f"Recorded sales amount across rows with valid Amount: {total_sales:,.2f}")
print(f"Highest-sales gender code: {gender_sales_rank.index[0]} ({gender_sales_rank.iloc[0]:,.2f})")
print(f"Highest-sales age group: {age_sales_rank.index[0]} ({age_sales_rank.iloc[0]:,.2f})")
print(f"Highest-sales state: {state_sales_rank.index[0]} ({state_sales_rank.iloc[0]:,.2f})")
print(f"Highest-sales product category: {category_sales_rank.index[0]} ({category_sales_rank.iloc[0]:,.2f})")
print(f"Highest-sales occupation: {occupation_sales_rank.index[0]} ({occupation_sales_rank.iloc[0]:,.2f})")

## 9. Business interpretation and next steps

- Use the highest-contributing age groups, states, and product categories as starting points for more detailed customer and campaign analysis.
- Compare both sales amount and order counts; a high total can reflect more orders, higher order values, or both.
- Validate demographic patterns over multiple periods before using them to target campaigns.
- Confirm the source's definitions for `Amount`, `Orders`, and the marital-status codes before making operational decisions.
- Further analysis could include average order value, category-by-state comparisons, and a Power BI dashboard.

## Limitations

- This is descriptive analysis of the provided dataset, not a causal study.
- Twelve records have missing `Amount` values and are excluded only from sales-value calculations.
- The original source and data dictionary were not provided. Interpretations of encoded fields and currency units should be verified before publication.